# Official CODI sparse answer-aligned KV gradient signal on Kaggle

## Goal

Test whether useful KV supervision exists only as a sparse, consistently answer-aligned gradient component that is obscured by complete KV targets.

The notebook learns a frozen five-percent parameter-coordinate mask on a fresh calibration split, then compares sparse, full, random-sparse, shuffled-sparse, complement, and answer-only updates on separate update and validation splits.

Enable **Internet** and a **T4 GPU**. Use **Save Version → Save & Run All** after pinning the commit. The browser may be closed once the committed Kaggle run begins.

## 1. Required input and run configuration

Attach the exported dataset from `kaggle_official_codi_kv_target_utility.ipynb`. It must contain the completed `official_codi_kv_target_utility/kind_seed3` directory. This experiment verifies that prior negative gate and excludes all of its discovery and validation question groups.

Run setup once with `RUN_COMMIT = "main"`, copy the printed immutable commit, restart, and use that hash for the final saved run.

In [ ]:
REPO_URL = "https://github.com/0x0shephard/latent-reasoning.git"
RUN_COMMIT = "main"  # Pin the printed immutable commit before Save Version.
REPO_DIR = "/kaggle/working/latent-reasoning"

# Leave empty to auto-discover the attached completed kind_seed3 artifact.
PRIOR_TARGET_UTILITY_INPUT = ""
# Optional attached output from an interrupted run of this notebook.
RESUME_INPUT = ""
# Optional exact passed official-CODI full-GSM8K summary.json.
REPRODUCTION_SUMMARY_INPUT = ""
RUN_REPRODUCTION_GATE_IF_MISSING = True

RUN_SMOKE = True
RUN_FULL_EXPERIMENT = True

EXAMPLES_PER_SPLIT = 128
SMOKE_EXAMPLES_PER_SPLIT = 8
BATCH_SIZE = 4
SEED = 5
PRIMARY_KIND = "key"
SPARSITY = 0.05
MINIMUM_POSITIVE_FRACTION = 0.60
RANDOM_MASK_SEED = 20260729
METRIC = "l1"
KV_WEIGHT = 1.0
RELATIVE_UPDATE_NORM = 1e-4
PRECISION = "float32"
BOOTSTRAP_SAMPLES = 10000
BOOTSTRAP_SEED = 0

UPLOAD_AS_KAGGLE_DATASET = False
KAGGLE_DATASET_HANDLE = "jonraza15/official-codi-kv-gradient-signal"

## 2. Set up the pinned repository

In [ ]:
import datetime
import hashlib
import json
import os
import pathlib
import shutil
import subprocess
import sys

os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("Hugging Face authentication: Kaggle secret loaded")
except Exception:
    print("Hugging Face authentication: public access")

repo = pathlib.Path(REPO_DIR)
if not (repo / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
subprocess.run(["git", "-C", REPO_DIR, "fetch", "origin"], check=True)
target = f"origin/{RUN_COMMIT}" if RUN_COMMIT == "main" else RUN_COMMIT
subprocess.run(["git", "-C", REPO_DIR, "checkout", "--detach", target], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-official-codi.txt")],
    check=True,
)
os.chdir(REPO_DIR)
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Checked out:", commit)
if RUN_COMMIT == "main":
    print("PIN RUN_COMMIT BEFORE THE FINAL SAVE VERSION RUN:", commit)

## 3. Verify the Kaggle GPU and implementation

In [ ]:
import torch
import transformers

assert torch.cuda.is_available(), "Enable a Kaggle GPU accelerator"
capability = torch.cuda.get_device_capability(0)
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("GPU:", torch.cuda.get_device_name(0), "capability:", capability)
assert capability >= (7, 0), (
    f"GPU capability {capability} is unsupported. Select a T4 and restart."
)
subprocess.run(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_official_codi.py",
        "tests/test_official_codi_kv.py",
        "tests/test_official_codi_target_utility.py",
        "tests/test_kv_gradient_signal.py",
        "tests/test_official_codi_kv_gradient_signal_analysis.py",
    ],
    cwd=REPO_DIR,
    check=True,
)

## 4. Locate the completed kind-level target-utility gate

The attached artifact is read-only. Duplicate work-tree and export copies are accepted only when their request hashes are identical.

In [ ]:
if PRIOR_TARGET_UTILITY_INPUT:
    PRIOR_TARGET_UTILITY_DIR = pathlib.Path(PRIOR_TARGET_UTILITY_INPUT)
else:
    manifests = list(
        pathlib.Path("/kaggle/input").rglob(
            "official_codi_kv_target_utility/kind_seed3/run_manifest.json"
        )
    )
    assert manifests, (
        "Attach the exported target-utility dataset or set "
        "PRIOR_TARGET_UTILITY_INPUT explicitly."
    )
    request_hashes = {
        json.loads(path.read_text()).get("request_sha256") for path in manifests
    }
    assert len(request_hashes) == 1, f"Attached kind screens disagree: {manifests}"
    PRIOR_TARGET_UTILITY_DIR = sorted(
        {path.parent for path in manifests},
        key=lambda path: (len(path.parts), path.as_posix()),
    )[0]

prior_summary = json.loads((PRIOR_TARGET_UTILITY_DIR / "summary.json").read_text())
prior_manifest = json.loads((PRIOR_TARGET_UTILITY_DIR / "run_manifest.json").read_text())
assert prior_manifest["state"] == "complete"
assert len(prior_manifest["completed_batches"]) == 32
assert prior_summary["screen_status"] == "no_helpful_target_family_at_this_granularity"
assert prior_summary["classifications"] == {
    "key_all": "neutral_or_inconclusive_target_family",
    "value_all": "neutral_or_inconclusive_target_family",
}
print("Prior target-utility directory:", PRIOR_TARGET_UTILITY_DIR)
print("Prior gate:", prior_summary["screen_status"])
print("Prior classifications:", prior_summary["classifications"])

## 5. Prepare durable paths, logging, and optional resume

In [ ]:
WORK_OUTPUT_ROOT = repo / "outputs" / "official_codi_kv_gradient_signal"
WORK_LOG_ROOT = repo / "logs" / "official_codi_kv_gradient_signal"
VALIDATION_ROOT = repo / "outputs" / "official_codi_gpt2"
for path in (WORK_OUTPUT_ROOT, WORK_LOG_ROOT, VALIDATION_ROOT):
    path.mkdir(parents=True, exist_ok=True)

def run_persisted(command, log_name):
    log_path = WORK_LOG_ROOT / log_name
    print("Starting:", " ".join(map(str, command)), flush=True)
    print("Persistent session log:", log_path, flush=True)
    with log_path.open("a", encoding="utf-8", buffering=1) as log:
        log.write(
            f"\n=== {datetime.datetime.now(datetime.timezone.utc).isoformat()} "
            f"{' '.join(map(str, command))} ===\n"
        )
        process = subprocess.Popen(
            command,
            cwd=REPO_DIR,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
        return_code = process.wait()
        log.flush()
    if return_code != 0:
        raise RuntimeError(f"Command failed with exit code {return_code}; inspect {log_path}")
    return log_path

if RESUME_INPUT:
    resume_root = pathlib.Path(RESUME_INPUT)
    manifests = list(
        resume_root.rglob("official_codi_kv_gradient_signal/full_seed5/run_manifest.json")
    )
    assert manifests, "No full_seed5 gradient-signal run was found in RESUME_INPUT"
    hashes = {json.loads(path.read_text()).get("request_sha256") for path in manifests}
    assert len(hashes) == 1, f"RESUME_INPUT contains incompatible runs: {manifests}"
    source_root = sorted(
        {path.parents[1] for path in manifests},
        key=lambda path: (len(path.parts), path.as_posix()),
    )[0]
    shutil.copytree(source_root, WORK_OUTPUT_ROOT, dirs_exist_ok=True)
    print("Restored gradient-signal outputs from:", source_root)
else:
    print("Starting without a previous gradient-signal output")

## 6. Locate or create the official-CODI reproduction gate

In [ ]:
EXPECTED_CHECKPOINT_REVISION = "fd641b3d3edc59e4f534b55588e906588c9e36bb"

def is_passed_reproduction(path):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        return False
    return (
        payload.get("accuracy_gate", {}).get("status") == "passed"
        and payload.get("evaluated_counts", {}).get("gsm8k") == 1319
        and payload.get("checkpoint_revision") == EXPECTED_CHECKPOINT_REVISION
    )

if REPRODUCTION_SUMMARY_INPUT:
    REPRODUCTION_SUMMARY = pathlib.Path(REPRODUCTION_SUMMARY_INPUT)
    assert is_passed_reproduction(REPRODUCTION_SUMMARY)
else:
    candidates = [
        path for root in (pathlib.Path("/kaggle/input"), VALIDATION_ROOT)
        for path in root.rglob("summary.json")
        if is_passed_reproduction(path)
    ]
    if candidates:
        REPRODUCTION_SUMMARY = sorted(candidates, key=lambda path: path.as_posix())[0]
    else:
        assert RUN_REPRODUCTION_GATE_IF_MISSING
        run_persisted(
            [
                sys.executable, "-u", "-m", "src.eval.official_codi",
                "--config", "configs/official_codi_gpt2.yaml",
                "--datasets", "gsm8k",
                "--limit", "0",
                "--device", "cuda",
                "--output-dir", str(VALIDATION_ROOT),
            ],
            "official_codi_gsm8k_gate.log",
        )
        candidates = [
            path for path in VALIDATION_ROOT.rglob("summary.json")
            if is_passed_reproduction(path)
        ]
        assert len(candidates) == 1, f"Expected one passed summary, found {candidates}"
        REPRODUCTION_SUMMARY = candidates[0]
print("Reproduction summary:", REPRODUCTION_SUMMARY)

## 7. Build the fixed experiment command

In [ ]:
def gradient_signal_command(output_dir, examples_per_split):
    return [
        sys.executable, "-u", "scripts/run_official_codi_kv_gradient_signal.py",
        "--config", "configs/official_codi_gpt2.yaml",
        "--reproduction-summary", str(REPRODUCTION_SUMMARY),
        "--prior-target-utility-dir", str(PRIOR_TARGET_UTILITY_DIR),
        "--output-dir", str(output_dir),
        "--examples-per-split", str(examples_per_split),
        "--batch-size", str(BATCH_SIZE),
        "--metric", METRIC,
        "--kv-weight", str(KV_WEIGHT),
        "--sparsity", str(SPARSITY),
        "--minimum-positive-fraction", str(MINIMUM_POSITIVE_FRACTION),
        "--random-mask-seed", str(RANDOM_MASK_SEED),
        "--relative-update-norm", str(RELATIVE_UPDATE_NORM),
        "--precision", PRECISION,
        "--device", "cuda",
        "--seed", str(SEED),
        "--primary-kind", PRIMARY_KIND,
        "--bootstrap-samples", str(BOOTSTRAP_SAMPLES),
        "--bootstrap-seed", str(BOOTSTRAP_SEED),
    ]

## 8. Run an 8+8+8 example smoke test

The smoke test verifies mask fitting, random-mask cardinality, energy matching, all five conditions for keys and values, stateless updates, and report generation. Its statistical result is diagnostic only.

In [ ]:
from IPython.display import Markdown, display

SMOKE_ROOT = WORK_OUTPUT_ROOT / "smoke_seed5"
if RUN_SMOKE:
    run_persisted(
        gradient_signal_command(SMOKE_ROOT, SMOKE_EXAMPLES_PER_SPLIT),
        "smoke_seed5.log",
    )
    smoke_manifest = json.loads((SMOKE_ROOT / "run_manifest.json").read_text())
    assert smoke_manifest["state"] == "complete"
    assert len(smoke_manifest["completed_batches"]) == 2
    assert (SMOKE_ROOT / "mask_artifact.pt").is_file()
    display(Markdown((SMOKE_ROOT / "report.md").read_text()))
else:
    print("Smoke test skipped")

## 9. Run the full fresh three-split experiment

The full run uses 128 calibration, 128 update, and 128 validation examples. All normalized question groups are mutually disjoint and disjoint from the previous kind-level screen.

In [ ]:
FULL_ROOT = WORK_OUTPUT_ROOT / "full_seed5"
if RUN_FULL_EXPERIMENT:
    run_persisted(
        gradient_signal_command(FULL_ROOT, EXAMPLES_PER_SPLIT),
        "full_seed5.log",
    )
    full_manifest = json.loads((FULL_ROOT / "run_manifest.json").read_text())
    full_report = json.loads((FULL_ROOT / "summary.json").read_text())
    assert full_manifest["state"] == "complete"
    assert len(full_manifest["completed_batches"]) == EXAMPLES_PER_SPLIT // BATCH_SIZE
    assert full_report["evaluated_validation_examples"] == EXAMPLES_PER_SPLIT
    display(Markdown((FULL_ROOT / "report.md").read_text()))
else:
    assert (FULL_ROOT / "summary.json").is_file(), (
        "RUN_FULL_EXPERIMENT=False requires a restored completed full_seed5 run"
    )
    full_report = json.loads((FULL_ROOT / "summary.json").read_text())

## 10. Check the mask and decision gate

In [ ]:
print("FINAL GATE:", full_report["gate"])
print("PRIMARY KIND:", full_report["primary_kind"])
for kind, summary in full_report["mask_summaries"].items():
    print(
        kind,
        "selected=", summary["selected_coordinates"],
        "realized_sparsity=", f"{summary['realized_sparsity']:.4%}",
        "eligible=", summary["eligible_coordinates"],
    )
for kind, payload in full_report["by_kind"].items():
    print(kind, payload["classification"], payload["criteria"])

if full_report["gate"] == "primary_sparse_component_only_supported":
    print("DECISION: proceed to a frozen-mask exact causal and later training gate.")
elif full_report["gate"] == "primary_sparse_component_supported_not_only":
    print("DECISION: sparse signal is supported, but the claim that it is the only useful component is not.")
else:
    print("DECISION: close this coordinatewise answer-alignment definition; do not train with it.")

## 11. Build the durable Kaggle export

The export contains the frozen masks, all paired batch records, reports, logs, the pinned commit, and checksums. Model downloads and Hugging Face caches are excluded.

In [ ]:
EXPORT_ROOT = pathlib.Path("/kaggle/working/official_codi_kv_gradient_signal_export")
if EXPORT_ROOT.exists():
    shutil.rmtree(EXPORT_ROOT)
export_repo = EXPORT_ROOT / "latent-reasoning"
shutil.copytree(WORK_OUTPUT_ROOT, export_repo / "outputs" / "official_codi_kv_gradient_signal")
shutil.copytree(WORK_LOG_ROOT, export_repo / "logs" / "official_codi_kv_gradient_signal")
(EXPORT_ROOT / "RUN_COMMIT.txt").write_text(commit + "\n")
(EXPORT_ROOT / "PRIOR_TARGET_UTILITY_REQUEST.txt").write_text(
    prior_manifest["request_sha256"] + "\n"
)

files = sorted(path for path in EXPORT_ROOT.rglob("*") if path.is_file())
checksums = []
for path in files:
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksums.append(f"{digest}  {path.relative_to(EXPORT_ROOT).as_posix()}")
(EXPORT_ROOT / "SHA256SUMS.txt").write_text("\n".join(checksums) + "\n")
all_files = [path for path in EXPORT_ROOT.rglob("*") if path.is_file()]
print("Export root:", EXPORT_ROOT)
print("Files:", len(all_files))
print("Size:", sum(path.stat().st_size for path in all_files) / 2**20, "MiB")
print("Use Save Version with outputs enabled.")

## 12. Optional direct dataset upload

In [ ]:
if UPLOAD_AS_KAGGLE_DATASET:
    import kagglehub
    kagglehub.dataset_upload(
        KAGGLE_DATASET_HANDLE,
        str(EXPORT_ROOT),
        version_notes=f"Official CODI sparse KV gradient signal at {commit}",
    )
    print("Dataset upload complete:", KAGGLE_DATASET_HANDLE)
else:
    print("Direct upload skipped. Save this notebook version with outputs enabled.")

## Next steps

- A negative primary gate closes this coordinatewise answer-alignment definition.
- `primary_sparse_component_supported_not_only` supports sparse signal but not the word *only*.
- `primary_sparse_component_only_supported` permits a separate exact frozen-mask causal test.
- No result from this notebook alone authorizes expensive distillation training.